In [1]:

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Kaggle input paths
DATA_DIR = (
    "/kaggle/input/datasets/drrajgauravmishra/spatiotemporal-gnn-kaggle-data/Spatiotemporal_GNN_Kaggle_Data"
)

PREVIOUS_NOTEBOOK_DIR = (
    "/kaggle/input/notebooks/drrajgauravmishra/03-spatial-encoder-comparison-gcn-gat-gatv2-ipynb"
)

EMBEDDING_DIR = os.path.join(
    PREVIOUS_NOTEBOOK_DIR,
    "spatial_encoder_results"
)

CHECKPOINT_DIR = os.path.join(
    EMBEDDING_DIR,
    "checkpoints"
)

# New embedding files from Notebook 04
NEW_EMBEDDING_DIR = (
    "/kaggle/input/notebooks/drrajgauravmishra/04-gatv2-spatial-embedding-extraction-ipynb"
)

# Output directory
OUTPUT_DIR = "/kaggle/working/gatv2_gru_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Device:", DEVICE)
print("\nChecking dataset files:")

for filename in [
    "training_annual.npz",
    "validation_annual.npz",
    "testing_annual.npz",
    "static_spatial_graph_verified.pt"
]:
    path = os.path.join(DATA_DIR, filename)
    print(filename, ":", os.path.exists(path))

print("\nChecking GATv2 checkpoints:")

for seed in [42, 123, 2023]:
    filename = f"GATv2_seed_{seed}.pt"
    path = os.path.join(CHECKPOINT_DIR, filename)
    print(filename, ":", os.path.exists(path))


print("\nChecking complete embeddings:")

for seed in [42, 123, 2023]:

    filename = f"GATv2_seed_{seed}_embeddings_2000_2024.npy"

    # Search within Notebook 04 output directory
    matching_paths = []

    for root, dirs, files in os.walk(NEW_EMBEDDING_DIR):
        if filename in files:
            matching_paths.append(os.path.join(root, filename))

    if matching_paths:
        path = matching_paths[0]
        embeddings = np.load(path, mmap_mode="r")

        print(f"\nSeed {seed}:")
        print("Found:", path)
        print("Shape:", embeddings.shape)
        print("Status:", "PASS" if embeddings.shape == (25, 5040, 256) else "CHECK")

    else:
        print(f"\nSeed {seed}: FILE NOT FOUND")



Device: cuda

Checking dataset files:
training_annual.npz : True
validation_annual.npz : True
testing_annual.npz : True
static_spatial_graph_verified.pt : True

Checking GATv2 checkpoints:
GATv2_seed_42.pt : True
GATv2_seed_123.pt : True
GATv2_seed_2023.pt : True

Checking complete embeddings:

Seed 42:
Found: /kaggle/input/notebooks/drrajgauravmishra/04-gatv2-spatial-embedding-extraction-ipynb/gatv2_complete_embeddings/GATv2_seed_42_embeddings_2000_2024.npy
Shape: (25, 5040, 256)
Status: PASS

Seed 123:
Found: /kaggle/input/notebooks/drrajgauravmishra/04-gatv2-spatial-embedding-extraction-ipynb/gatv2_complete_embeddings/GATv2_seed_123_embeddings_2000_2024.npy
Shape: (25, 5040, 256)
Status: PASS

Seed 2023:
Found: /kaggle/input/notebooks/drrajgauravmishra/04-gatv2-spatial-embedding-extraction-ipynb/gatv2_complete_embeddings/GATv2_seed_2023_embeddings_2000_2024.npy
Shape: (25, 5040, 256)
Status: PASS


In [2]:

# Load annual forecasting datasets

train_data = np.load(
    os.path.join(DATA_DIR, "training_annual.npz")
)

val_data = np.load(
    os.path.join(DATA_DIR, "validation_annual.npz")
)

test_data = np.load(
    os.path.join(DATA_DIR, "testing_annual.npz")
)

print("Dataset keys:")
print("Training:", train_data.files)
print("Validation:", val_data.files)
print("Testing:", test_data.files)


# Locate and load complete GATv2 embeddings

gatv2_embeddings = {}

for seed in [42, 123, 2023]:

    filename = f"GATv2_seed_{seed}_embeddings_2000_2024.npy"

    embedding_path = None

    for root, dirs, files in os.walk(NEW_EMBEDDING_DIR):
        if filename in files:
            embedding_path = os.path.join(root, filename)
            break

    if embedding_path is None:
        raise FileNotFoundError(filename)

    gatv2_embeddings[seed] = np.load(embedding_path)

    print(
        f"Seed {seed}:",
        gatv2_embeddings[seed].shape
    )


print("\nAll datasets and embeddings loaded successfully.")


Dataset keys:
Training: ['X', 'Y', 'Y_mask', 'input_indices', 'target_indices', 'target_years']
Validation: ['X', 'Y', 'Y_mask', 'input_indices', 'target_indices', 'target_years']
Testing: ['X', 'Y', 'Y_mask', 'input_indices', 'target_indices', 'target_years']
Seed 42: (25, 5040, 256)
Seed 123: (25, 5040, 256)
Seed 2023: (25, 5040, 256)

All datasets and embeddings loaded successfully.


In [3]:

# Prepare GATv2 embedding sequences for forecasting

def prepare_embedding_sequences(split_data, embeddings):

    X_indices = split_data["input_indices"]
    sequences = []

    for indices in X_indices:
        # Select the 10 annual spatial embedding snapshots
        sequence = embeddings[indices]

        sequences.append(sequence)

    return np.stack(sequences)


# Prepare sequences for each seed and data split

embedding_sequences = {}

for seed in [42, 123, 2023]:

    embedding_sequences[seed] = {
        "train": prepare_embedding_sequences(
            train_data, gatv2_embeddings[seed]
        ),
        "validation": prepare_embedding_sequences(
            val_data, gatv2_embeddings[seed]
        ),
        "test": prepare_embedding_sequences(
            test_data, gatv2_embeddings[seed]
        )
    }

    print(f"\nSeed: {seed}")

    for split in ["train", "validation", "test"]:
        print(
            split,
            embedding_sequences[seed][split].shape
        )

print("\nEmbedding sequence preparation completed.")



Seed: 42
train (8, 10, 5040, 256)
validation (6, 10, 5040, 256)
test (2, 10, 5040, 256)

Seed: 123
train (8, 10, 5040, 256)
validation (6, 10, 5040, 256)
test (2, 10, 5040, 256)

Seed: 2023
train (8, 10, 5040, 256)
validation (6, 10, 5040, 256)
test (2, 10, 5040, 256)

Embedding sequence preparation completed.


In [4]:

# GATv2-GRU forecasting architecture

class GATv2GRUForecaster(nn.Module):

    def __init__(
        self,
        input_dim=256,
        hidden_dim=128,
        num_layers=1,
        output_dim=2,
        dropout=0.2
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):

        batch_size, time_steps, num_nodes, num_features = x.shape

        # Process each spatial node's temporal sequence independently
        x = x.permute(0, 2, 1, 3)

        x = x.reshape(
            batch_size * num_nodes,
            time_steps,
            num_features
        )

        _, hidden = self.gru(x)

        representation = self.dropout(hidden[-1])

        output = self.fc(representation)

        return output.reshape(batch_size, num_nodes, 2)


# Masked MSE loss
# Select valid targets before calculating errors to avoid NaN propagation.

def masked_mse(pred, target, mask):

    valid_pred = pred[mask]
    valid_target = target[mask]

    return torch.mean((valid_pred - valid_target) ** 2)


# Architecture verification
model = GATv2GRUForecaster(
    input_dim=256,
    hidden_dim=128,
    dropout=0.2
).to(DEVICE)

# Test with one sample and 32 nodes
test_input = torch.tensor(
    embedding_sequences[42]["train"][0:1, :, :32, :],
    dtype=torch.float32,
    device=DEVICE
)

with torch.no_grad():
    test_output = model(test_input)

print("Device:", DEVICE)
print("Input shape:", tuple(test_input.shape))
print("Output shape:", tuple(test_output.shape))
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Architecture verification completed.")


Device: cuda
Input shape: (1, 10, 32, 256)
Output shape: (1, 32, 2)
Model parameters: 148482
Architecture verification completed.


In [5]:

# Inspect forecasting targets and masks

for split_name, split_data in [
    ("Training", train_data),
    ("Validation", val_data),
    ("Testing", test_data)
]:

    Y = split_data["Y"]
    mask = split_data["Y_mask"]

    print(f"\n{split_name}")
    print("-" * 45)

    print("Target shape:", Y.shape)
    print("Target dtype:", Y.dtype)
    print("Mask shape:", mask.shape)
    print("Mask dtype:", mask.dtype)

    print("Target NaN count:", np.isnan(Y).sum())
    print("Target infinity count:", np.isinf(Y).sum())

    print("Valid mask entries:", np.sum(mask))
    print("Total mask entries:", mask.size)

    for variable, name in enumerate(["Rainfall", "Temperature"]):
        print(
            f"{name} valid entries:",
            np.sum(mask[..., variable])
        )


# Check embedding memory footprint

print("\nEmbedding memory requirements")
print("-" * 45)

for seed in [42, 123, 2023]:
    arr = gatv2_embeddings[seed]
    print(
        f"Seed {seed}:",
        round(arr.nbytes / (1024**2), 2),
        "MB"
    )



Training
---------------------------------------------
Target shape: (8, 5040, 2)
Target dtype: float32
Mask shape: (8, 5040, 2)
Mask dtype: bool
Target NaN count: 7189
Target infinity count: 0
Valid mask entries: 73451
Total mask entries: 80640
Rainfall valid entries: 37888
Temperature valid entries: 35563

Validation
---------------------------------------------
Target shape: (6, 5040, 2)
Target dtype: float32
Mask shape: (6, 5040, 2)
Mask dtype: bool
Target NaN count: 5437
Target infinity count: 0
Valid mask entries: 55043
Total mask entries: 60480
Rainfall valid entries: 28416
Temperature valid entries: 26627

Testing
---------------------------------------------
Target shape: (2, 5040, 2)
Target dtype: float32
Mask shape: (2, 5040, 2)
Mask dtype: bool
Target NaN count: 1736
Target infinity count: 0
Valid mask entries: 18424
Total mask entries: 20160
Rainfall valid entries: 9472
Temperature valid entries: 8952

Embedding memory requirements
----------------------------------------

In [6]:

# Final target-mask and GPU memory validation

for split_name, split_data in [
    ("Training", train_data),
    ("Validation", val_data),
    ("Testing", test_data)
]:

    Y = split_data["Y"]
    mask = split_data["Y_mask"]

    # Every valid target must be finite
    assert np.isfinite(Y[mask]).all(), (
        f"Non-finite valid targets in {split_name}"
    )

    # Every invalid target should be non-finite
    assert np.isnan(Y[~mask]).all(), (
        f"Unexpected values in invalid targets: {split_name}"
    )

    print(f"{split_name}: target-mask validation passed")


# GPU information

if torch.cuda.is_available():

    print("\nGPU:", torch.cuda.get_device_name(0))

    total_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024**3)
    )

    free_memory, allocated_memory = torch.cuda.mem_get_info()

    print("Total GPU memory:", round(total_memory, 2), "GB")
    print("Currently free:", round(free_memory / (1024**3), 2), "GB")
    print("Currently allocated:", round(allocated_memory / (1024**3), 2), "GB")

print("\nReady for forecasting training.")


Training: target-mask validation passed
Validation: target-mask validation passed
Testing: target-mask validation passed

GPU: Tesla T4
Total GPU memory: 14.56 GB
Currently free: 14.42 GB
Currently allocated: 14.56 GB

Ready for forecasting training.


In [7]:

if torch.cuda.is_available():

    free_memory, total_memory = torch.cuda.mem_get_info()

    allocated_memory = torch.cuda.memory_allocated()
    reserved_memory = torch.cuda.memory_reserved()

    print("GPU:", torch.cuda.get_device_name(0))
    print("Total memory:", round(total_memory / 1024**3, 2), "GB")
    print("Free memory:", round(free_memory / 1024**3, 2), "GB")
    print("Allocated memory:", round(allocated_memory / 1024**3, 2), "GB")
    print("Reserved memory:", round(reserved_memory / 1024**3, 2), "GB")


GPU: Tesla T4
Total memory: 14.56 GB
Free memory: 14.42 GB
Allocated memory: 0.01 GB
Reserved memory: 0.02 GB


In [8]:

import copy
import time
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def train_gatv2_gru(
    seed,
    hidden_dim,
    max_epochs,
    train_embeddings,
    val_embeddings,
    train_targets,
    train_masks,
    val_targets,
    val_masks,
    learning_rate=0.001,
    weight_decay=1e-5,
    dropout=0.2,
    patience=30
):

    set_seed(seed)

    model = GATv2GRUForecaster(
        input_dim=256,
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    # Convert data to tensors
    X_train = torch.tensor(
        train_embeddings,
        dtype=torch.float32,
        device=DEVICE
    )

    Y_train = torch.tensor(
        train_targets,
        dtype=torch.float32,
        device=DEVICE
    )

    M_train = torch.tensor(
        train_masks,
        dtype=torch.bool,
        device=DEVICE
    )

    X_val = torch.tensor(
        val_embeddings,
        dtype=torch.float32,
        device=DEVICE
    )

    Y_val = torch.tensor(
        val_targets,
        dtype=torch.float32,
        device=DEVICE
    )

    M_val = torch.tensor(
        val_masks,
        dtype=torch.bool,
        device=DEVICE
    )

    best_val_loss = float("inf")
    best_epoch = 0
    best_state = None

    train_history = []
    val_history = []

    patience_counter = 0

    start_time = time.time()

    for epoch in range(1, max_epochs + 1):

        # Training
        model.train()
        optimizer.zero_grad()

        train_predictions = model(X_train)

        train_loss = masked_mse(
            train_predictions,
            Y_train,
            M_train
        )

        train_loss.backward()
        optimizer.step()

        # Validation
        model.eval()

        with torch.no_grad():

            val_predictions = model(X_val)

            val_loss = masked_mse(
                val_predictions,
                Y_val,
                M_val
            )

        train_value = train_loss.item()
        val_value = val_loss.item()

        train_history.append(train_value)
        val_history.append(val_value)

        # Early stopping
        if val_value < best_val_loss:

            best_val_loss = val_value
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:
            patience_counter += 1

        if epoch % 25 == 0:

            print(
                f"Epoch {epoch:03d} | "
                f"Train: {train_value:.6f} | "
                f"Validation: {val_value:.6f} | "
                f"Best: {best_val_loss:.6f}"
            )

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    elapsed_time = time.time() - start_time

    # Restore best validation checkpoint
    model.load_state_dict(best_state)

    return {
        "model": model,
        "best_state": best_state,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "train_history": train_history,
        "val_history": val_history,
        "training_time": elapsed_time
    }


print("GATv2-GRU training function defined successfully.")


GATv2-GRU training function defined successfully.


In [9]:

# Pilot run: Seed 42, hidden dimension 64, 5 epochs

pilot_result = train_gatv2_gru(
    seed=42,
    hidden_dim=64,
    max_epochs=5,

    train_embeddings=embedding_sequences[42]["train"],
    val_embeddings=embedding_sequences[42]["validation"],

    train_targets=train_data["Y"],
    train_masks=train_data["Y_mask"],

    val_targets=val_data["Y"],
    val_masks=val_data["Y_mask"],

    learning_rate=0.001,
    weight_decay=1e-5,
    dropout=0.2,
    patience=30
)

print("\nPILOT TRAINING RESULTS")
print("-" * 45)

print("Best epoch:", pilot_result["best_epoch"])
print("Best validation loss:", pilot_result["best_val_loss"])
print("Training time:", round(pilot_result["training_time"], 2), "seconds")

print("Training losses:", pilot_result["train_history"])
print("Validation losses:", pilot_result["val_history"])

print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

print("\nPilot run completed.")



PILOT TRAINING RESULTS
---------------------------------------------
Best epoch: 5
Best validation loss: 442.53399658203125
Training time: 1.44 seconds
Training losses: [485.6184387207031, 478.1415100097656, 470.7134704589844, 463.3508605957031, 455.99481201171875]
Validation losses: [471.6924743652344, 464.3658447265625, 457.0685729980469, 449.79156494140625, 442.53399658203125]

GPU memory allocated: 0.02 GB

Pilot run completed.


In [10]:

# Numerical scale diagnostic

def describe_array(name, array):

    array = np.asarray(array)

    print(f"\n{name}")
    print("-" * 45)
    print("Shape:", array.shape)
    print("Min:", np.nanmin(array))
    print("Max:", np.nanmax(array))
    print("Mean:", np.nanmean(array))
    print("Std:", np.nanstd(array))
    print("NaN count:", np.isnan(array).sum())


# 1. Spatial embedding statistics
describe_array(
    "GATv2 embeddings - Seed 42",
    gatv2_embeddings[42]
)

# 2. Target statistics
describe_array(
    "Training targets - Rainfall",
    train_data["Y"][..., 0]
)

describe_array(
    "Training targets - Temperature",
    train_data["Y"][..., 1]
)

# 3. Model prediction statistics
pilot_result["model"].eval()

with torch.no_grad():

    pilot_predictions = pilot_result["model"](
        torch.tensor(
            embedding_sequences[42]["train"],
            dtype=torch.float32,
            device=DEVICE
        )
    ).cpu().numpy()

describe_array(
    "Pilot predictions - Rainfall",
    pilot_predictions[..., 0]
)

describe_array(
    "Pilot predictions - Temperature",
    pilot_predictions[..., 1]
)

print("\nDiagnostic completed.")



GATv2 embeddings - Seed 42
---------------------------------------------
Shape: (25, 5040, 256)
Min: 0.0
Max: 3.875455
Mean: 0.18945068
Std: 0.23706189
NaN count: 0

Training targets - Rainfall
---------------------------------------------
Shape: (8, 5040)
Min: 0.0
Max: 31.854368
Mean: 3.0588877
Std: 2.1566586
NaN count: 2432

Training targets - Temperature
---------------------------------------------
Shape: (8, 5040)
Min: 19.288761
Max: 35.0763
Mean: 31.361946
Std: 2.9900622
NaN count: 4757

Pilot predictions - Rainfall
---------------------------------------------
Shape: (8, 5040)
Min: -0.2886752
Max: -0.027871601
Mean: -0.12257542
Std: 0.039902303
NaN count: 0

Pilot predictions - Temperature
---------------------------------------------
Shape: (8, 5040)
Min: 1.2369145
Max: 2.3149977
Mean: 1.3407406
Std: 0.09908922
NaN count: 0

Diagnostic completed.


In [11]:

# ============================================================
# TARGET NORMALIZATION
# Training data only; validation and testing are not used.
# ============================================================

target_names = ["Rainfall", "Maximum Temperature"]

Y_train = train_data["Y"]
Y_val = val_data["Y"]
Y_test = test_data["Y"]

Y_train_mask = train_data["Y_mask"]
Y_val_mask = val_data["Y_mask"]
Y_test_mask = test_data["Y_mask"]

target_mean = np.zeros(2, dtype=np.float32)
target_std = np.zeros(2, dtype=np.float32)

for j in range(2):
    valid_values = Y_train[:, :, j][Y_train_mask[:, :, j]]

    target_mean[j] = valid_values.mean()
    target_std[j] = valid_values.std()

    if target_std[j] == 0:
        target_std[j] = 1.0

print("\nTRAINING-ONLY NORMALIZATION")
print("-" * 50)

for j, name in enumerate(target_names):
    print(
        f"{name}: mean={target_mean[j]:.6f}, "
        f"std={target_std[j]:.6f}"
    )


def normalize_targets(Y, mask):
    Y_norm = np.zeros_like(Y, dtype=np.float32)

    for j in range(2):
        valid = mask[:, :, j]
        Y_norm[:, :, j][valid] = (
            Y[:, :, j][valid] - target_mean[j]
        ) / target_std[j]

    return Y_norm


Y_train_norm = normalize_targets(Y_train, Y_train_mask)
Y_val_norm = normalize_targets(Y_val, Y_val_mask)
Y_test_norm = normalize_targets(Y_test, Y_test_mask)

print("\nNormalized target shapes:")
print("Training:", Y_train_norm.shape)
print("Validation:", Y_val_norm.shape)
print("Testing:", Y_test_norm.shape)



TRAINING-ONLY NORMALIZATION
--------------------------------------------------
Rainfall: mean=3.058888, std=2.156659
Maximum Temperature: mean=31.361950, std=2.990062

Normalized target shapes:
Training: (8, 5040, 2)
Validation: (6, 5040, 2)
Testing: (2, 5040, 2)


In [12]:

# ============================================================
# GATv2-GRU: NORMALIZED-TARGET PILOT RUN
# ============================================================


pilot_result = train_gatv2_gru(
    seed=42,
    hidden_dim=64,
    max_epochs=5,
    train_embeddings=embedding_sequences[42]["train"],
    val_embeddings=embedding_sequences[42]["validation"],
    train_targets=Y_train_norm,
    train_masks=Y_train_mask,
    val_targets=Y_val_norm,
    val_masks=Y_val_mask,
    learning_rate=0.001,
    weight_decay=1e-5,
    dropout=0.2,
    patience=30
)

print("\nNORMALIZED-TARGET PILOT RESULTS")
print("-" * 50)
print("Best epoch:", pilot_result["best_epoch"])
print("Best validation loss:", pilot_result["best_val_loss"])
print("Training time:", round(pilot_result["training_time"], 2), "seconds")
#print("Training losses:", pilot_result["train_losses"])
#print("Validation losses:", pilot_result["val_losses"])



NORMALIZED-TARGET PILOT RESULTS
--------------------------------------------------
Best epoch: 5
Best validation loss: 0.6531309485435486
Training time: 0.59 seconds


In [13]:
print("Available result keys:")
print(pilot_result.keys())

Available result keys:
dict_keys(['model', 'best_state', 'best_epoch', 'best_val_loss', 'train_history', 'val_history', 'training_time'])


In [14]:
print("NORMALIZED-TARGET PILOT LOSS HISTORY")
print("-" * 50)

print("Training losses:", pilot_result["train_history"])
print("Validation losses:", pilot_result["val_history"])

print("\nBest epoch:", pilot_result["best_epoch"])
print("Best validation loss:", pilot_result["best_val_loss"])

NORMALIZED-TARGET PILOT LOSS HISTORY
--------------------------------------------------
Training losses: [1.0485097169876099, 0.9533759951591492, 0.8629859089851379, 0.7828612923622131, 0.7062768340110779]
Validation losses: [0.9910992980003357, 0.898468554019928, 0.8114445805549622, 0.7295930981636047, 0.6531309485435486]

Best epoch: 5
Best validation loss: 0.6531309485435486


In [15]:

# ============================================================
# GATv2-GRU HYPERPARAMETER TUNING
# Hidden dimensions: 64, 128
# Seeds: 42, 123, 2023
# Maximum epochs: 300
# ============================================================

import pandas as pd
import time

hidden_dimensions = [64, 128]
seeds = [42, 123, 2023]
max_epochs = 300

tuning_results = []
tuning_models = {}

total_runs = len(hidden_dimensions) * len(seeds)
run_number = 0

for hidden_dim in hidden_dimensions:

    for seed in seeds:

        run_number += 1

        print("\n" + "=" * 65)
        print(f"RUN {run_number}/{total_runs}")
        print(f"Hidden dimension: {hidden_dim} | Seed: {seed}")
        print("=" * 65)

        result = train_gatv2_gru(
            seed=seed,
            hidden_dim=hidden_dim,
            max_epochs=max_epochs,
            train_embeddings=embedding_sequences[seed]["train"],
            val_embeddings=embedding_sequences[seed]["validation"],
            train_targets=Y_train_norm,
            train_masks=Y_train_mask,
            val_targets=Y_val_norm,
            val_masks=Y_val_mask,
            learning_rate=0.001,
            weight_decay=1e-5,
            dropout=0.2,
            patience=30
        )

        tuning_models[(hidden_dim, seed)] = result

        tuning_results.append({
            "hidden_dim": hidden_dim,
            "seed": seed,
            "best_epoch": result["best_epoch"],
            "best_val_loss": result["best_val_loss"],
            "training_time": result["training_time"]
        })

        print(f"\nBest epoch: {result['best_epoch']}")
        print(f"Best validation loss: {result['best_val_loss']:.6f}")
        print(f"Training time: {result['training_time']:.2f} seconds")

# ============================================================
# SUMMARY
# ============================================================

tuning_df = pd.DataFrame(tuning_results)

print("\n\nINDIVIDUAL RUN RESULTS")
print("=" * 65)
display(tuning_df)

tuning_summary = (
    tuning_df.groupby("hidden_dim")
    .agg(
        mean_val_loss=("best_val_loss", "mean"),
        std_val_loss=("best_val_loss", "std"),
        mean_best_epoch=("best_epoch", "mean"),
        mean_training_time=("training_time", "mean")
    )
    .reset_index()
)

print("\nHYPERPARAMETER SUMMARY")
print("=" * 65)
display(tuning_summary)

# Save results
tuning_df.to_csv(
    os.path.join(OUTPUT_DIR, "gatv2_gru_tuning_individual.csv"),
    index=False
)

tuning_summary.to_csv(
    os.path.join(OUTPUT_DIR, "gatv2_gru_tuning_summary.csv"),
    index=False
)

print("\nTuning results saved successfully.")



RUN 1/6
Hidden dimension: 64 | Seed: 42
Epoch 025 | Train: 0.211450 | Validation: 0.168885 | Best: 0.168885
Epoch 050 | Train: 0.186938 | Validation: 0.151341 | Best: 0.151341
Epoch 075 | Train: 0.180014 | Validation: 0.145379 | Best: 0.145379
Epoch 100 | Train: 0.175921 | Validation: 0.142180 | Best: 0.142180
Epoch 125 | Train: 0.174113 | Validation: 0.140117 | Best: 0.140117
Epoch 150 | Train: 0.171557 | Validation: 0.138710 | Best: 0.138710
Epoch 175 | Train: 0.169782 | Validation: 0.138235 | Best: 0.138235
Epoch 200 | Train: 0.169022 | Validation: 0.137867 | Best: 0.137641
Epoch 225 | Train: 0.167191 | Validation: 0.137826 | Best: 0.137497
Epoch 250 | Train: 0.165813 | Validation: 0.137535 | Best: 0.137260
Epoch 275 | Train: 0.164381 | Validation: 0.137227 | Best: 0.136993
Epoch 300 | Train: 0.162604 | Validation: 0.137239 | Best: 0.136989

Best epoch: 287
Best validation loss: 0.136989
Training time: 36.50 seconds

RUN 2/6
Hidden dimension: 64 | Seed: 123
Epoch 025 | Train: 0.217

,hidden_dim,seed,best_epoch,best_val_loss,training_time
0,64,42,287,0.136989,36.498654
1,64,123,300,0.150096,37.498687
2,64,2023,295,0.136190,39.919386
3,128,42,168,0.136783,62.895549
4,128,123,124,0.151521,49.089767
5,128,2023,196,0.135871,71.829624



HYPERPARAMETER SUMMARY


,hidden_dim,mean_val_loss,std_val_loss,mean_best_epoch,mean_training_time
0,64,0.141091,0.007808,294.000000,37.972243
1,128,0.141392,0.008784,162.666667,61.271647



Tuning results saved successfully.


In [16]:

# ============================================================
# GATv2-GRU: FINAL EXPERIMENTAL TRAINING
# ============================================================

import os
import json
import torch
import pandas as pd

FINAL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR, "final_experimental_runs"
)

FINAL_CHECKPOINT_DIR = os.path.join(
    FINAL_OUTPUT_DIR, "checkpoints"
)

os.makedirs(FINAL_CHECKPOINT_DIR, exist_ok=True)

hidden_dimensions = [64, 128]
epoch_limits = [300, 500]
seeds = [42, 123, 2023]

learning_rate = 0.001
weight_decay = 1e-5
dropout = 0.2
patience = 30

final_results = []
final_models = {}

total_runs = (
    len(hidden_dimensions) *
    len(epoch_limits) *
    len(seeds)
)

run_number = 0

for hidden_dim in hidden_dimensions:

    for max_epochs in epoch_limits:

        for seed in seeds:

            run_number += 1

            print("\n" + "=" * 70)
            print(f"FINAL EXPERIMENT: {run_number}/{total_runs}")
            print(f"Hidden dimension : {hidden_dim}")
            print(f"Maximum epochs   : {max_epochs}")
            print(f"Random seed      : {seed}")
            print("=" * 70)

            result = train_gatv2_gru(
                seed=seed,
                hidden_dim=hidden_dim,
                max_epochs=max_epochs,

                train_embeddings=embedding_sequences[seed]["train"],
                val_embeddings=embedding_sequences[seed]["validation"],

                train_targets=Y_train_norm,
                train_masks=Y_train_mask,

                val_targets=Y_val_norm,
                val_masks=Y_val_mask,

                learning_rate=learning_rate,
                weight_decay=weight_decay,
                dropout=dropout,
                patience=patience
            )

            run_key = (hidden_dim, max_epochs, seed)

            final_models[run_key] = result

            # Save best model checkpoint
            checkpoint_name = (
                f"GATv2_GRU_H{hidden_dim}_"
                f"E{max_epochs}_Seed{seed}.pt"
            )

            checkpoint_path = os.path.join(
                FINAL_CHECKPOINT_DIR,
                checkpoint_name
            )

            torch.save({
                "model_state_dict": result["best_state"],
                "hidden_dim": hidden_dim,
                "max_epochs": max_epochs,
                "seed": seed,
                "best_epoch": result["best_epoch"],
                "best_val_loss": result["best_val_loss"],
                "learning_rate": learning_rate,
                "weight_decay": weight_decay,
                "dropout": dropout,
                "patience": patience,
                "target_mean": target_mean,
                "target_std": target_std
            }, checkpoint_path)

            # Record results
            final_results.append({
                "hidden_dim": hidden_dim,
                "max_epochs": max_epochs,
                "seed": seed,
                "best_epoch": result["best_epoch"],
                "best_val_loss": result["best_val_loss"],
                "training_time": result["training_time"],
                "checkpoint": checkpoint_name
            })

            print("\nBest epoch:", result["best_epoch"])
            print(
                "Best validation loss:",
                f"{result['best_val_loss']:.6f}"
            )
            print(
                "Training time:",
                f"{result['training_time']:.2f} seconds"
            )

            print("Checkpoint saved:", checkpoint_name)


# ============================================================
# INDIVIDUAL EXPERIMENT RESULTS
# ============================================================

final_results_df = pd.DataFrame(final_results)

print("\n\nINDIVIDUAL EXPERIMENT RESULTS")
print("=" * 70)
display(final_results_df)


# ============================================================
# GROUPED SUMMARY
# ============================================================

final_summary_df = (
    final_results_df
    .groupby(["hidden_dim", "max_epochs"])
    .agg(
        mean_val_loss=("best_val_loss", "mean"),
        std_val_loss=("best_val_loss", "std"),
        mean_best_epoch=("best_epoch", "mean"),
        std_best_epoch=("best_epoch", "std"),
        mean_training_time=("training_time", "mean")
    )
    .reset_index()
)

print("\nFINAL EXPERIMENTAL SUMMARY")
print("=" * 70)
display(final_summary_df)


# ============================================================
# SAVE RESULTS
# ============================================================

final_results_df.to_csv(
    os.path.join(
        FINAL_OUTPUT_DIR,
        "gatv2_gru_final_individual_results.csv"
    ),
    index=False
)

final_summary_df.to_csv(
    os.path.join(
        FINAL_OUTPUT_DIR,
        "gatv2_gru_final_summary.csv"
    ),
    index=False
)

normalization_metadata = {
    "target_names": target_names,
    "target_mean": target_mean.tolist(),
    "target_std": target_std.tolist()
}

with open(
    os.path.join(
        FINAL_OUTPUT_DIR,
        "target_normalization_metadata.json"
    ),
    "w"
) as f:
    json.dump(normalization_metadata, f, indent=4)

print("\n" + "=" * 70)
print("ALL FINAL EXPERIMENTS COMPLETED")
print("Total runs:", len(final_results_df))
print("Checkpoints:", FINAL_CHECKPOINT_DIR)
print("Results:", FINAL_OUTPUT_DIR)
print("=" * 70)



FINAL EXPERIMENT: 1/12
Hidden dimension : 64
Maximum epochs   : 300
Random seed      : 42
Epoch 025 | Train: 0.211450 | Validation: 0.168885 | Best: 0.168885
Epoch 050 | Train: 0.186938 | Validation: 0.151341 | Best: 0.151341
Epoch 075 | Train: 0.180014 | Validation: 0.145379 | Best: 0.145379
Epoch 100 | Train: 0.175921 | Validation: 0.142180 | Best: 0.142180
Epoch 125 | Train: 0.174113 | Validation: 0.140117 | Best: 0.140117
Epoch 150 | Train: 0.171557 | Validation: 0.138710 | Best: 0.138710
Epoch 175 | Train: 0.169782 | Validation: 0.138235 | Best: 0.138235
Epoch 200 | Train: 0.169022 | Validation: 0.137867 | Best: 0.137641
Epoch 225 | Train: 0.167191 | Validation: 0.137826 | Best: 0.137497
Epoch 250 | Train: 0.165813 | Validation: 0.137535 | Best: 0.137260
Epoch 275 | Train: 0.164381 | Validation: 0.137227 | Best: 0.136993
Epoch 300 | Train: 0.162604 | Validation: 0.137239 | Best: 0.136989

Best epoch: 287
Best validation loss: 0.136989
Training time: 39.51 seconds
Checkpoint saved

,hidden_dim,max_epochs,seed,best_epoch,best_val_loss,training_time,checkpoint
0,64,300,42,287,0.136989,39.513886,GATv2_GRU_H64_E300_Seed42.pt
1,64,300,123,300,0.150096,39.590709,GATv2_GRU_H64_E300_Seed123.pt
2,64,300,2023,295,0.136190,39.577118,GATv2_GRU_H64_E300_Seed2023.pt
3,64,500,42,364,0.134461,51.977700,GATv2_GRU_H64_E500_Seed42.pt
4,64,500,123,455,0.141469,63.962961,GATv2_GRU_H64_E500_Seed123.pt
5,64,500,2023,409,0.132209,58.015155,GATv2_GRU_H64_E500_Seed2023.pt
6,128,300,42,168,0.136783,63.235769,GATv2_GRU_H128_E300_Seed42.pt
7,128,300,123,124,0.151521,48.790108,GATv2_GRU_H128_E300_Seed123.pt
8,128,300,2023,196,0.135871,71.927519,GATv2_GRU_H128_E300_Seed2023.pt
9,128,500,42,168,0.136783,62.677457,GATv2_GRU_H128_E500_Seed42.pt



FINAL EXPERIMENTAL SUMMARY


,hidden_dim,max_epochs,mean_val_loss,std_val_loss,mean_best_epoch,std_best_epoch,mean_training_time
0,64,300,0.141091,0.007808,294.000000,6.557439,39.560571
1,64,500,0.136046,0.004829,409.333333,45.500916,57.985272
2,128,300,0.141392,0.008784,162.666667,36.295087,61.317799
3,128,500,0.141392,0.008784,162.666667,36.295087,61.107671



ALL FINAL EXPERIMENTS COMPLETED
Total runs: 12
Checkpoints: /kaggle/working/gatv2_gru_results/final_experimental_runs/checkpoints
Results: /kaggle/working/gatv2_gru_results/final_experimental_runs


In [17]:

# ============================================================
# GATv2-GRU: FINAL TEST EVALUATION
# All 12 saved experimental checkpoints
# Metrics calculated in original physical units
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

test_results = []

variables = ["Rainfall", "Maximum Temperature"]

# Original, unnormalized test targets
Y_test_original = test_data["Y"]
Y_test_mask = test_data["Y_mask"]

for hidden_dim in [64, 128]:

    for max_epochs in [300, 500]:

        for seed in [42, 123, 2023]:

            print("\n" + "=" * 65)
            print(
                f"Testing: Hidden={hidden_dim}, "
                f"Epochs={max_epochs}, Seed={seed}"
            )
            print("=" * 65)

            checkpoint_name = (
                f"GATv2_GRU_H{hidden_dim}_"
                f"E{max_epochs}_Seed{seed}.pt"
            )

            checkpoint_path = os.path.join(
                FINAL_CHECKPOINT_DIR,
                checkpoint_name
            )

            checkpoint = torch.load(
                checkpoint_path,
                map_location=DEVICE,
                weights_only=False
            )

            model = GATv2GRUForecaster(
                input_dim=256,
                hidden_dim=hidden_dim,
                dropout=0.2
            ).to(DEVICE)

            model.load_state_dict(
                checkpoint["model_state_dict"]
            )

            model.eval()

            X_test_tensor = torch.tensor(
                embedding_sequences[seed]["test"],
                dtype=torch.float32,
                device=DEVICE
            )

            with torch.no_grad():
                predictions_norm = (
                    model(X_test_tensor)
                    .cpu()
                    .numpy()
                )

            # Convert predictions back to original units
            predictions_original = (
                predictions_norm * target_std.reshape(1, 1, 2)
                + target_mean.reshape(1, 1, 2)
            )

            # Calculate metrics separately for each variable
            for j, variable in enumerate(variables):

                valid_mask = Y_test_mask[:, :, j]

                y_true = Y_test_original[:, :, j][valid_mask]
                y_pred = predictions_original[:, :, j][valid_mask]

                mae = mean_absolute_error(y_true, y_pred)
                rmse = np.sqrt(mean_squared_error(y_true, y_pred))
                r2 = r2_score(y_true, y_pred)

                test_results.append({
                    "hidden_dim": hidden_dim,
                    "max_epochs": max_epochs,
                    "seed": seed,
                    "variable": variable,
                    "MAE": mae,
                    "RMSE": rmse,
                    "R2": r2,
                    "valid_nodes": len(y_true)
                })

                print(
                    f"{variable}: "
                    f"MAE={mae:.6f}, "
                    f"RMSE={rmse:.6f}, "
                    f"R2={r2:.6f}"
                )


# ============================================================
# INDIVIDUAL TEST RESULTS
# ============================================================

test_results_df = pd.DataFrame(test_results)

print("\n\nINDIVIDUAL TEST RESULTS")
print("=" * 65)
display(test_results_df)


# ============================================================
# MEAN AND STANDARD DEVIATION ACROSS THREE SEEDS
# ============================================================

test_summary_df = (
    test_results_df
    .groupby(["hidden_dim", "max_epochs", "variable"])
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std")
    )
    .reset_index()
)

print("\nFINAL TEST SUMMARY")
print("=" * 65)
display(test_summary_df)


# ============================================================
# SAVE RESULTS
# ============================================================

test_results_df.to_csv(
    os.path.join(
        FINAL_OUTPUT_DIR,
        "gatv2_gru_test_individual_results.csv"
    ),
    index=False
)

test_summary_df.to_csv(
    os.path.join(
        FINAL_OUTPUT_DIR,
        "gatv2_gru_test_summary.csv"
    ),
    index=False
)

print("\nTest evaluation completed.")
print("Results saved in:", FINAL_OUTPUT_DIR)



Testing: Hidden=64, Epochs=300, Seed=42
Rainfall: MAE=0.654504, RMSE=0.935215, R2=0.806095
Maximum Temperature: MAE=0.470478, RMSE=0.644792, R2=0.944790

Testing: Hidden=64, Epochs=300, Seed=123
Rainfall: MAE=0.685782, RMSE=1.004380, R2=0.776353
Maximum Temperature: MAE=0.459452, RMSE=0.633630, R2=0.946685

Testing: Hidden=64, Epochs=300, Seed=2023
Rainfall: MAE=0.657023, RMSE=0.939880, R2=0.804156
Maximum Temperature: MAE=0.459226, RMSE=0.636266, R2=0.946241

Testing: Hidden=64, Epochs=500, Seed=42
Rainfall: MAE=0.646751, RMSE=0.922388, R2=0.811378
Maximum Temperature: MAE=0.469404, RMSE=0.640460, R2=0.945529

Testing: Hidden=64, Epochs=500, Seed=123
Rainfall: MAE=0.670835, RMSE=0.940733, R2=0.803800
Maximum Temperature: MAE=0.465339, RMSE=0.633418, R2=0.946721

Testing: Hidden=64, Epochs=500, Seed=2023
Rainfall: MAE=0.660717, RMSE=0.930859, R2=0.807897
Maximum Temperature: MAE=0.466002, RMSE=0.636606, R2=0.946183

Testing: Hidden=128, Epochs=300, Seed=42
Rainfall: MAE=0.651603, RMSE

,hidden_dim,max_epochs,seed,variable,MAE,RMSE,R2,valid_nodes
0,64,300,42,Rainfall,0.654504,0.935215,0.806095,9472
1,64,300,42,Maximum Temperature,0.470478,0.644792,0.944790,8952
2,64,300,123,Rainfall,0.685782,1.004380,0.776353,9472
3,64,300,123,Maximum Temperature,0.459452,0.633630,0.946685,8952
4,64,300,2023,Rainfall,0.657023,0.939880,0.804156,9472
5,64,300,2023,Maximum Temperature,0.459226,0.636266,0.946241,8952
6,64,500,42,Rainfall,0.646751,0.922388,0.811378,9472
7,64,500,42,Maximum Temperature,0.469404,0.640460,0.945529,8952
8,64,500,123,Rainfall,0.670835,0.940733,0.803800,9472
9,64,500,123,Maximum Temperature,0.465339,0.633418,0.946721,8952



FINAL TEST SUMMARY


,hidden_dim,max_epochs,variable,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,64,300,Maximum Temperature,0.463052,0.006432,0.638229,0.005834,0.945905,0.000991
1,64,300,Rainfall,0.665770,0.017377,0.959825,0.038656,0.795535,0.016640
2,64,500,Maximum Temperature,0.466915,0.002181,0.636828,0.003527,0.946144,0.000597
3,64,500,Rainfall,0.659434,0.012094,0.931326,0.009182,0.807692,0.003793
4,128,300,Maximum Temperature,0.464272,0.003768,0.647592,0.006891,0.944305,0.001182
5,128,300,Rainfall,0.660681,0.018961,0.957669,0.043290,0.796395,0.018622
6,128,500,Maximum Temperature,0.464272,0.003768,0.647592,0.006891,0.944305,0.001182
7,128,500,Rainfall,0.660681,0.018961,0.957669,0.043290,0.796395,0.018622



Test evaluation completed.
Results saved in: /kaggle/working/gatv2_gru_results/final_experimental_runs
